# 部署已训练的 SmolVLA 策略

<img src="./media/rollout3.gif" width="480" height="360">

在仿真中部署已训练策略。


In [2]:
!pip install transformers==4.50.3
!pip install num2words
!pip install accelerate
!pip install safetensors>=0.4.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 9.2 MB/s eta 0:00:00 0:00:01m
  Attempting uninstall: transformers
    Found existing installation: transformers 4.48.0
    Uninstalling transformers-4.48.0:
      Successfully uninstalled transformers-4.48.0

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13783 sha256=ef1ba085d70e4411967fffcddc818dc531cb337d06df7da9950b01795d4b3b20
  Stored in directory: /home/jeongeun/.cache/pip/wheels/fc/ab/d4/5da2067ac95b36618c629a5f93f809425700506f72c9732fac
Successfully built docopt

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To upd

### 【可选】下载数据集


In [ ]:
'''
If you want to use the collected dataset, please download it from Hugging Face.
'''
!git clone https://huggingface.co/datasets/datawhale-eai/datawhale_eai_pnp_language

## 第 2 步：训练模型


In [ ]:
!python train_model.py --config_path smolvla_datawhale_eai.yaml

## 第 3 步：部署


In [1]:
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
import numpy as np
from lerobot.common.datasets.utils import write_json, serialize_dict
from lerobot.common.policies.smolvla.configuration_smolvla import SmolVLAConfig
from lerobot.common.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.configs.types import FeatureType
from lerobot.common.datasets.factory import resolve_delta_timestamps
from lerobot.common.datasets.utils import dataset_to_policy_features
import torch
from PIL import Image
import torchvision

/home/jeongeun/.pyenv/versions/3.10.2/envs/lerobot/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = 'cuda'

In [3]:
try:
    dataset_metadata = LeRobotDatasetMetadata("datawhale_eai_pnp_language", root='./demo_data_language')
except:
    dataset_metadata = LeRobotDatasetMetadata("datawhale_eai_pnp_language", root='./datawhale_eai_pnp_language')
features = dataset_to_policy_features(dataset_metadata.features)
output_features = {key: ft for key, ft in features.items() if ft.type is FeatureType.ACTION}
input_features = {key: ft for key, ft in features.items() if key not in output_features}
# Policies are initialized with a configuration class, in this case `DiffusionConfig`. For this example,
# we'll just use the defaults and so no arguments other than input/output features need to be passed.
# Temporal ensemble to make smoother trajectory predictions
cfg = SmolVLAConfig(input_features=input_features, output_features=output_features, chunk_size= 5, n_action_steps=5)
delta_timestamps = resolve_delta_timestamps(cfg, dataset_metadata)

# We can now instantiate our policy with this config and the dataset stats.
policy = SmolVLAPolicy.from_pretrained('./ckpt/smolvla_datawhale_eai/checkpoints/last/pretrained_model', dataset_stats=dataset_metadata.stats)
# You can load the trained policy from hub if you don't have the resources to train it.
# policy = SmolVLAPolicy.from_pretrained("datawhale-eai/smolvla_datawhale_eai", config=cfg, dataset_stats=dataset_metadata.stats)
policy.to(device)




In [5]:
from mujoco_env.y_env2 import SimpleEnv2
xml_path = './asset/example_scene_y2.xml'
PnPEnv = SimpleEnv2(xml_path, action_type='joint_angle')


-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[31] n_qvel:[28] n_qacc:[28] n_ctrl:[10]
 integrator:[IMPLICITFAST]

n_body:[23]
 [0/23] [world] mass:[0.00]kg
 [1/23] [front_object_table] mass:[1.00]kg
 [2/23] [camera] mass:[0.00]kg
 [3/23] [camera2] mass:[0.00]kg
 [4/23] [camera3] mass:[0.00]kg
 [5/23] [link1] mass:[2.06]kg
 [6/23] [link2] mass:[3.68]kg
 [7/23] [link3] mass:[2.39]kg
 [8/23] [link4] mass:[1.40]kg
 [9/23] [link5] mass:[1.40]kg
 [10/23] [link6] mass:[0.65]kg
 [11/23] [camera_center] mass:[0.00]kg
 [12/23] [tcp_link] mass:[0.32]kg
 [13/23] [rh_p12_rn_r1] mass:[0.07]kg
 [14/23] [rh_p12_rn_r2] mass:[0.02]kg
 [15/23] [rh_p12_rn_l1] mass:[0.07]kg
 [16/23] [rh_p12_rn_l2] mass:[0.02]kg
 [17/23] [body_obj_mug_5] mass:[0.00]kg
 [18/23] [object_mug_5] mass:[0.08]kg
 [19/23] [body_obj_plate_11] mass:[0.00]kg
 [20/23] [object_plate_11] mass:[0.10]kg
 [21/23] [body_obj_mug_6] mass:[0.00]kg
 [22/23] [object_mug

In [6]:
from torchvision import transforms
# Approach 1: Using torchvision.transforms
def get_default_transform(image_size: int = 224):
    """
    Returns a torchvision transform that:
     Converts to a FloatTensor and scales pixel values [0,255] -> [0.0,1.0]
    """
    return transforms.Compose([
        transforms.ToTensor(),  # PIL [0–255] -> FloatTensor [0.0–1.0], shape C×H×W
    ])

In [7]:
step = 0
PnPEnv.reset(seed=0)
policy.reset()
policy.eval()
save_image = True
IMG_TRANSFORM = get_default_transform()
while PnPEnv.env.is_viewer_alive():
    PnPEnv.step_env()
    if PnPEnv.env.loop_every(HZ=20):
        # Check if the task is completed
        success = PnPEnv.check_success()
        if success:
            print('Success')
            # Reset the environment and action queue
            policy.reset()
            PnPEnv.reset()
            step = 0
            save_image = False
        # Get the current state of the environment
        state = PnPEnv.get_joint_state()[:6]
        # Get the current image from the environment
        image, wirst_image = PnPEnv.grab_image()
        image = Image.fromarray(image)
        image = image.resize((256, 256))
        image = IMG_TRANSFORM(image)
        wrist_image = Image.fromarray(wirst_image)
        wrist_image = wrist_image.resize((256, 256))
        wrist_image = IMG_TRANSFORM(wrist_image)
        data = {
            'observation.state': torch.tensor([state]).to(device),
            'observation.image': image.unsqueeze(0).to(device),
            'observation.wrist_image': wrist_image.unsqueeze(0).to(device),
            'task': [PnPEnv.instruction],
        }
        # Select an action
        action = policy.select_action(data)
        action = action[0,:7].cpu().detach().numpy()
        # Take a step in the environment
        _ = PnPEnv.step(action)
        PnPEnv.render()
        step += 1
        success = PnPEnv.check_success()
        if success:
            print('Success')
            break

DONE INITIALIZATION


/tmp/ipykernel_3326535/3859105902.py:30: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  'observation.state': torch.tensor([state]).to(device),


Success
DONE INITIALIZATION
Success
DONE INITIALIZATION


In [8]:
# policy.push_to_hub(
#     repo_id='datawhale-eai/smolvla_datawhale_eai',
#     commit_message='Add trained policy for PnP task',
# )

model.safetensors: 100%|██████████| 1.20G/1.20G [02:26<00:00, 8.19MB/s]


CommitInfo(commit_url='https://huggingface.co/datawhale-eai/smolvla_datawhale_eai/commit/738e2cb9235562d58842ceb8a9469ae21278bb53', commit_message='Add trained policy for PnP task', commit_description='', oid='738e2cb9235562d58842ceb8a9469ae21278bb53', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datawhale-eai/smolvla_datawhale_eai', endpoint='https://huggingface.co', repo_type='model', repo_id='datawhale-eai/smolvla_datawhale_eai'), pr_revision=None, pr_num=None)

**从小步快跑的巨兽（$\pi_0$），走向精干灵巧的刺客（SmolVLA）。**

SmolVLA（"Smol" 是网络流行语，意思是“小巧可爱的”）直接继承了 $\pi_0$ 最核心的“双脑融合 + 流匹配”架构，但把底座全部换成了极度轻量化的组件。

我们先用一张图来看看它的内部构造，再来解答“为什么它这么小却这么强”。

### 📊 SmolVLA 架构工作流

代码段

```mermaid
graph TD
    %% Inputs
    subgraph Inputs [输入层]
        IMG[视觉观察 Image 16x16 Patch]
        TXT[语言指令 Language Prompt]
        STATE[本体状态 State 32-dim]
        NOISE[噪声动作 Action Noise 32-dim]
        TIME[时间步 t]
    end

    %% Vision-Language Backbone
    subgraph Brain [SmolVLM 大脑：轻量级语义理解]
        ViT[SmolVLM Vision Transformer 12层, 768维]
        PROJ[简单MLP投影器 Connector 768->960维]
        LLM_MAIN[Llama 主语言模型 16层, 960维]
    end

    %% Action Expert
    subgraph Cerebellum [Llama Expert 小脑：高效运动控制]
        STATE_PROJ[State Proj 32->960]
        ACT_PROJ[Action Proj 32->720]
        TIME_MLP[Time MLP 1440->720]
        EXPERT_LLM[Llama 动作专家 16层, 720维]
    end

    %% Flow Matching Output
    subgraph Outputs [输出层]
        OUT_PROJ[Action Out Proj 720->32]
        FLOW[VLA Flow Matching]
        ACT_CLEAN[平滑连续动作 Continuous Actions]
    end

    %% Connections
    IMG --> ViT --> PROJ --> LLM_MAIN
    TXT --> LLM_MAIN

    STATE --> STATE_PROJ --> EXPERT_LLM
    NOISE --> ACT_PROJ --> EXPERT_LLM
    TIME --> TIME_MLP --> EXPERT_LLM
    
    %% Cross-communication
    LLM_MAIN -.->|特征条件化| EXPERT_LLM

    EXPERT_LLM --> OUT_PROJ --> FLOW
    FLOW -->|迭代去噪| ACT_CLEAN
```

------

### 🆚 核心对比：SmolVLA vs. $\pi_0$

从你提供的网络结构打印信息，我们可以清晰地看到 SmolVLA 对 $\pi_0$ 进行了全面的“瘦身”：

| **模块**           | **π0 (重型装甲)**     | **SmolVLA (轻量级刺客)**     | **瘦身效果与意义**                                           |
| ------------------ | --------------------- | ---------------------------- | ------------------------------------------------------------ |
| **视觉编码器**     | SigLIP (27层, 1152维) | SmolVLM Vision (12层, 768维) | 计算量锐减。去掉了冗余的深层视觉特征，只保留抓取所需的关键空间信息。 |
| **主语言模型**     | Gemma (18层, 2048维)  | Llama架构 (16层, 960维)      | 维度砍半。主脑不再追求“无所不知”的互联网百科，而是专注于“看图说话”的具身指令理解。 |
| **动作专家(小脑)** | Gemma (18层, 1024维)  | Llama架构 (16层, 720维)      | 专家网络更紧凑，大大降低了推理延迟（Latency）。              |

------

### 🚀 为什么 SmolVLA 这么小，效果却这么好？

你可能会问：模型变小了，智商不就低了吗？为什么在具身智能里它还能表现出色？这得益于以下几个关键技术与理念：

#### 1. 守住了“版本答案”：双脑架构 + Flow Matching

模型可以变小，但**范式不能降级**。

SmolVLA 完全保留了 $\pi_0$ 最精髓的 `vlm_with_expert` 设计。这意味着它依然避免了“语义对齐税”（大脑和小脑分开工作），并且依然使用 `VLAFlowMatching` 来生成连续的、极其平滑的动作。只要保留了流匹配（Flow Matching），即使专家网络的特征维度从 1024 降到了 720，它依然能精准地拟合复杂的机器人动作分布。

#### 2. Llama 架构的底层优化 (GQA 的魔力)

仔细看你提供的 `lm_expert` 代码：

Plaintext

```
(q_proj): Linear(in_features=720, out_features=960, bias=False)
(k_proj): Linear(in_features=720, out_features=320, bias=False)
```

你会发现，Query ($q$) 的维度是 960，而 Key ($k$) 和 Value ($v$) 的维度只有 320！

这是典型的 **GQA (Grouped Query Attention, 分组查询注意力)** 机制。相比于传统的注意力机制，GQA 极大地减少了模型在推理时 KV Cache 占用的显存，提升了计算速度。对于需要高频输出动作指令（比如 50Hz）的机器人来说，**推理速度比绝对的参数量更重要**。

#### 3. “数据质量”战胜了“模型体积”

在 AI 界有一个共识（比如微软的 Phi 系列小模型验证的）：**Small models + High quality data = Big performance**。

具身智能领域目前最大的瓶颈是“高质量的机器人操作数据不足”，而不是“模型参数量不够”。SmolVLA 证明了，只要给它喂入清晰、干净的遥操作（Teleoperation）数据，一个小巧的 Llama 网络足以学会“如何完美地煎一个鸡蛋”或“折叠一件衣服”。巨大的模型（如千亿参数的 LLM）在干这些“肌肉活”时，大部分算力其实是处于闲置或浪费状态的。

#### 4. 模态连接器的极简主义

代码中 `SmolVLMConnector` 是一个极其简单的 `Linear(12288 -> 960)`。它没有使用复杂的重采样器（Resampler）或 Q-Former。这种“直筒子”设计让视觉特征无损、快速地拍脸传给语言模型，非常适合需要快速反应的控制任务。

------

### 💡 总结

如果说 $\pi_0$ 是一辆装备了顶级雷达和超算中心的重型坦克，用来探索具身智能的**能力天花板**；

那么 **SmolVLA** 就是一辆经过极其专业改装的拉力赛车。它卸掉了不必要的重量（超大维度的 LLM 和深层 ViT），换上了最高效的引擎（Llama GQA），并保留了最顶级的底盘调校（Flow Matching 和 Expert 架构）。

它在具身智能界引起轰动的原因很简单：**它让普通实验室甚至个人开发者，用一两张消费级显卡（比如 RTX 4090），就能跑得起甚至训练得起最前沿的机器人策略模型！**